## Common nodes and 8-neighbour graph

In [ ]:
from pathlib import Path
import json
import math
import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window, bounds as window_bounds
from shapely import wkt
from shapely.geometry import box
from shapely.prepared import prep
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from IPython.display import display

### Paths and raster inputs

In [ ]:
PROJECT_ROOT = Path.home() / "padova_data"
SOURCE = PROJECT_ROOT / "gnn_outputs" / "grid_90m_20220617_v1"
OUT = PROJECT_ROOT / "gnn_outputs" / "grid90_three_dates_preparation_v1"
OUT.mkdir(parents=True, exist_ok=True)

REFERENCE_LST = PROJECT_ROOT / "rasters" / "20220617" / "align" / "align_lst.tif"
RASTER_DIRS = {
    "2023-08-15": PROJECT_ROOT / "rasters" / "20230815" / "align",
    "2024-08-25": PROJECT_ROOT / "rasters" / "20240825" / "align",
}

FILES = {
    "lst": "align_lst.tif",
    "ndvi": "align_ndvi.tif",
    "ndbi": "align_ndbi.tif",
    "ndwi": "align_ndwi.tif",
    "albedo": "align_albedo.tif",
    "svf": "align_svf.tif",
    "bcr": "align_BCR.tif",
    "tcd": "align_TCD.tif",
    "pop": "align_pop.tif",
}

FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "svf", "bcr", "tcd", "pop"]
PUBLIC_FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"]
VARIABLES = ["lst"] + FEATURES
DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]

MIN_COVERAGE = 0.80
RANGES = {
    "ndvi": (-1, 1), "ndbi": (-1, 1), "ndwi": (-1, 1),
    "albedo": (0, 1), "svf": (0, 1), "bcr": (0, 1),
    "tcd": (0, 100), "pop": (0, np.inf),
}
STORAGE_CODES = [-9999, -19999]

# Preserve the reference node order.
nodes = pd.read_csv(SOURCE / "grid_node_table.csv", dtype={"grid_id": str})
nodes = nodes.sort_values("local_node_id").reset_index(drop=True)
nodes["grid_row"] = nodes["grid_row"].astype(int)
nodes["grid_col"] = nodes["grid_col"].astype(int)

geometry_table = pd.read_csv(SOURCE / "grid_geometry_wkt.csv", dtype={"grid_id": str})
geometry_table = geometry_table.set_index("grid_id").reindex(nodes["grid_id"])

geometries = [wkt.loads(t) for t in geometry_table["study_part_wkt"]]
geometry_areas = np.array([g.area for g in geometries])

# Read reference raster info
with rasterio.open(REFERENCE_LST) as src:
    REF_CRS = src.crs
    REF_TRANSFORM = src.transform
    REF_HEIGHT = src.height
    REF_WIDTH = src.width

if REF_TRANSFORM.b != 0 or REF_TRANSFORM.d != 0 or REF_TRANSFORM.a <= 0 or REF_TRANSFORM.e >= 0:
    raise ValueError("Expected a north-up reference grid.")

PIXEL_AREA = abs(REF_TRANSFORM.a * REF_TRANSFORM.e)
print(f"Reference nodes: {len(nodes)}, raster size: {REF_HEIGHT} x {REF_WIDTH}")

In [ ]:
# Precompute overlap areas between 90 m cells and 30 m pixels
pixel_links = []
pixel_height = abs(REF_TRANSFORM.e)

for geom in geometries:
    xmin, ymin, xmax, ymax = geom.bounds
    col_start = max(0, math.floor((xmin - REF_TRANSFORM.c) / REF_TRANSFORM.a))
    col_stop = min(REF_WIDTH, math.ceil((xmax - REF_TRANSFORM.c) / REF_TRANSFORM.a))
    row_start = max(0, math.floor((REF_TRANSFORM.f - ymax) / pixel_height))
    row_stop = min(REF_HEIGHT, math.ceil((REF_TRANSFORM.f - ymin) / pixel_height))

    prepared = prep(geom)
    indices, weights = [], []

    for r in range(row_start, row_stop):
        for c in range(col_start, col_stop):
            pixel = box(*window_bounds(Window(c, r, 1, 1), REF_TRANSFORM))
            if prepared.covers(pixel):
                area = PIXEL_AREA
            elif prepared.intersects(pixel):
                area = geom.intersection(pixel).area
            else:
                continue

            if area > 1e-8:
                indices.append(r * REF_WIDTH + c)
                weights.append(area)

    pixel_links.append((np.array(indices, dtype=np.int64), np.array(weights, dtype=np.float64)))

print("Pixel overlap cache ready.")

In [ ]:
# Helper functions for raster cleaning and quality flags
cleaning_rows = []

def read_clean_raster(path, variable, date):
    with rasterio.open(path) as src:
        if (src.crs != REF_CRS or src.shape != (REF_HEIGHT, REF_WIDTH)
                or not src.transform.almost_equals(REF_TRANSFORM)):
            raise ValueError(f"Raster is not aligned to the reference grid: {path}")
        a = src.read(1, masked=True).astype("float64").filled(np.nan)
        a = np.array(a, dtype=np.float64)

        # Remove invalid storage codes
        bad_storage = np.isfinite(a) & ((a < -1e30) | np.isin(a, STORAGE_CODES))
        removed_storage = int(bad_storage.sum())
        a[bad_storage | ~np.isfinite(a)] = np.nan

        a = a * src.scales[0] + src.offsets[0]
        a[~np.isfinite(a)] = np.nan

    # Remove out-of-range values
    removed_range = 0
    if variable in RANGES:
        low, high = RANGES[variable]
        bad_range = np.isfinite(a) & ((a < low) | (a > high))
        removed_range = int(bad_range.sum())
        a[bad_range] = np.nan

    finite = np.isfinite(a)
    zero_count = int((a[finite] == 0).sum())



    if not finite.any():
        raise ValueError(f"No valid pixels in {path}")
    if variable == "lst" and zero_count:
        if any((a.ravel()[idx] == 0).any() for idx, _ in pixel_links):
            raise ValueError(f"Zero-valued LST overlaps a study cell on {date}.")

    cleaning_rows.append({
        "date": date,
        "variable": variable,
        "valid_pixels": int(finite.sum()),
        "removed_storage_codes": removed_storage,
        "removed_out_of_range": removed_range,
        "zero_pixels": zero_count,
        "min": float(a[finite].min()),
        "median": float(np.median(a[finite])),
        "max": float(a[finite].max()),
    })
    return a.ravel()

def add_quality_flags(df):
    out = df.copy()
    feat_valid = np.isfinite(out[FEATURES].to_numpy(dtype=float)).all(axis=1)
    feat_cov_ok = (out[[f"{v}_coverage" for v in FEATURES]] >= MIN_COVERAGE).all(axis=1).to_numpy()
    out["inputs_ok"] = feat_valid & feat_cov_ok

    out["target_ok"] = np.isfinite(out["lst"].to_numpy(dtype=float)) & (out["lst_coverage"].to_numpy() >= MIN_COVERAGE)
    out["sample_ok"] = out["inputs_ok"] & out["target_ok"]
    return out

In [ ]:
# Compute area-weighted features for each date
identity_cols = ["grid_id", "grid_row", "grid_col", "x", "y", "area_m2"]

baseline = nodes[identity_cols + VARIABLES + [f"{v}_coverage" for v in VARIABLES]].copy()
baseline["date"] = "2022-06-17"
tables = {"2022-06-17": add_quality_flags(baseline)}

for date, folder in RASTER_DIRS.items():
    print("Processing date:", date)
    table = nodes[identity_cols].copy()
    table["date"] = date

    for var in VARIABLES:
        flat = read_clean_raster(folder / FILES[var], var, date)
        means = np.full(len(nodes), np.nan, dtype=np.float64)
        coverage = np.zeros(len(nodes), dtype=np.float64)




        for i, (indices, weights) in enumerate(pixel_links):
            vals = flat[indices]
            valid = np.isfinite(vals)
            valid_area = weights[valid].sum()
            coverage[i] = valid_area / geometry_areas[i]
            if valid_area > 0:
                means[i] = np.dot(vals[valid], weights[valid]) / valid_area

        table[var] = means
        table[f"{var}_coverage"] = np.clip(coverage, 0, 1)

    tables[date] = add_quality_flags(table)

# Keep nodes that are valid on all three dates
quality = nodes[["grid_id"]].copy()
for d in DATES:
    quality[f"sample_ok_{d}"] = tables[d]["sample_ok"].to_numpy()

common_mask = quality[[f"sample_ok_{d}" for d in DATES]].all(axis=1).to_numpy()
quality["common_valid"] = common_mask
quality.to_csv(OUT / "quality_by_grid_and_date.csv", index=False)




common_nodes = nodes.loc[common_mask, identity_cols].copy().reset_index(drop=True)
common_nodes.insert(0, "original_node_id", nodes.loc[common_mask, "local_node_id"].to_numpy())
common_nodes.insert(0, "local_node_id", np.arange(len(common_nodes)))
common_nodes.to_csv(OUT / "common_node_order.csv", index=False)

for d in DATES:
    tables[d].to_csv(OUT / f"grid_features_original_nodes_{d}.csv", index=False)
    common_df = tables[d].loc[common_mask].reset_index(drop=True)
    common_df.insert(0, "original_node_id", common_nodes["original_node_id"].to_numpy())
    common_df.insert(0, "local_node_id", np.arange(len(common_df)))
    common_df.to_csv(OUT / f"grid_node_table_{d}.csv", index=False)

print("Common valid nodes across all 3 dates:", len(common_nodes))

In [ ]:
# Build 8-neighbour graph and save summaries
lookup = {}
for i, row in common_nodes.iterrows():
    lookup[(int(row["grid_row"]), int(row["grid_col"]))] = i

offsets = [(0, 1), (1, -1), (1, 0), (1, 1)]
pairs = []
for (r, c), i in lookup.items():
    for dr, dc in offsets:
        j = lookup.get((r + dr, c + dc))
        if j is not None:
            pairs.append((i, j))

if pairs:
    undirected = np.array(pairs, dtype=np.int64).T
    edge_index = np.concatenate([undirected, undirected[::-1]], axis=1)
else:
    edge_index = np.empty((2, 0), dtype=np.int64)

n = len(common_nodes)
degree = np.bincount(edge_index[0], minlength=n)
adj = coo_matrix((np.ones(edge_index.shape[1], dtype=np.int8), (edge_index[0], edge_index[1])), shape=(n, n)).tocsr()
n_components, comp_labels = connected_components(adj, directed=False)
comp_sizes = np.bincount(comp_labels)

np.save(OUT / "edge_index_grid8_common.npy", edge_index)
np.save(OUT / "edge_weight_grid8_common.npy", np.ones(edge_index.shape[1], dtype=np.float32))

graph_summary = pd.DataFrame([{
    "nodes": n,
    "undirected_edges": len(pairs),
    "directed_edge_entries": edge_index.shape[1],
    "mean_degree": degree.mean(),
    "min_degree": degree.min(),
    "max_degree": degree.max(),
    "isolated_nodes": int((degree == 0).sum()),
    "components": int(n_components),
    "largest_component": int(comp_sizes.max()),
}])
graph_summary.to_csv(OUT / "graph_summary.csv", index=False)

# Save summary tables and config
date_summary = pd.DataFrame([
    {
        "date": d,
        "original_nodes": len(nodes),
        "input_valid": int(tables[d]["inputs_ok"].sum()),
        "label_valid": int(tables[d]["target_ok"].sum()),
        "sample_ok": int(tables[d]["sample_ok"].sum()),
        "common_nodes": n,
    }
    for d in DATES
])



##########
date_summary.to_csv(OUT / "date_quality_summary.csv", index=False)

coverage_rows = []
for d in DATES:
    for v in VARIABLES:
        cov = tables[d][f"{v}_coverage"]
        coverage_rows.append({
            "date": d, "variable": v,
            "nodes_below_80pct": int((cov < MIN_COVERAGE).sum()),
            "nodes_zero_coverage": int((cov == 0).sum()),
        })
pd.DataFrame(coverage_rows).to_csv(OUT / "coverage_summary.csv", index=False)
pd.DataFrame(cleaning_rows).to_csv(OUT / "raster_cleaning_summary.csv", index=False)

geometry_table.loc[common_nodes["grid_id"].tolist()].reset_index().to_csv(
    OUT / "common_grid_geometry_wkt.csv", index=False
)

config = {
    "dates": DATES,
    "source_grid_directory": str(SOURCE),
    "reference_raster": str(REFERENCE_LST),
    "raster_directories": {d: str(p) for d, p in RASTER_DIRS.items()},
    "features_all": FEATURES,
    "features_public": PUBLIC_FEATURES,
    "coverage_threshold": MIN_COVERAGE,
    "cell_size_m": 90,
    "original_nodes": len(nodes),
    "common_nodes": n,
    "graph": "8-neighbour grid, unit edge weights",
}
with open(OUT / "preparation_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

shutil.make_archive(str(OUT) + "_backup", "zip", root_dir=OUT)
display(date_summary)
display(graph_summary.round(4))

# Static GNN training package (standardized per split)

In [ ]:
BASE = PROJECT_ROOT / "gnn_outputs"
PREP = BASE / "grid90_three_dates_preparation_v1"
ORIGINAL = BASE / "static_grid90_20220617_v1"
OUT = BASE / "static_grid90_three_dates_v1"
OUT.mkdir(parents=True, exist_ok=True)

DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]
SPLITS = [42, 123, 2026, 3407, 777]
FEATURE_GROUPS = {
    "all": ["ndvi", "ndbi", "ndwi", "albedo", "svf", "bcr", "tcd", "pop"],
    "public": ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"],
}
MASK_KEYS = ["mask_train", "mask_valid", "mask_test"]

config = {
    "version": OUT.name, "dates": DATES, "split_seeds": SPLITS, "feature_groups": FEATURE_GROUPS,
    "graph": "grid8", "edge_weights": "all_one", "standardization": "train_nodes_only_per_date_and_split",
    "split_method": "restrict_original_masks_by_original_node_id", "standard_deviation_ddof": 0,
}
(OUT / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

# Save node order and graph
common = pd.read_csv(PREP / "common_node_order.csv", dtype={"grid_id": str}).sort_values("local_node_id").reset_index(drop=True)
original_ids = common["original_node_id"].to_numpy(dtype=np.int64)
n = len(common)
common.to_csv(OUT / "node_order.csv", index=False)

graph_dir = OUT / "graphs"
graph_dir.mkdir(exist_ok=True)
edge_index = np.load(PREP / "edge_index_grid8_common.npy")
edge_weight = np.load(PREP / "edge_weight_grid8_common.npy")
np.save(graph_dir / "edge_index_grid8.npy", edge_index.astype(np.int64))
np.save(graph_dir / "edge_weight_grid8.npy", edge_weight.astype(np.float32))

# Restrict original 2022 splits to the common nodes
mask_dir = OUT / "masks"
mask_dir.mkdir(exist_ok=True)
masks_by_split = {}
membership = common[["local_node_id", "original_node_id", "grid_id"]].copy()

for seed in SPLITS:
    with np.load(ORIGINAL / "all" / f"split_{seed}.npz") as saved:
        if not np.array_equal(saved["node_ids"][original_ids].astype(str), common["grid_id"]):
            raise ValueError(f"Node order differs in split {seed}.")
        masks = {k: saved[k][original_ids].astype(bool) for k in MASK_KEYS}
    if not (np.stack(list(masks.values())).sum(axis=0) == 1).all():
        raise ValueError(f"Invalid split membership for seed {seed}.")
    masks_by_split[seed] = masks

    np.savez_compressed(mask_dir / f"split_{seed}.npz", node_ids=common["grid_id"].to_numpy(dtype=str), **masks)

    subset = np.full(n, "", dtype="<U5")
    subset[masks["mask_train"]] = "train"
    subset[masks["mask_valid"]] = "val"
    subset[masks["mask_test"]] = "test"
    membership[f"split_{seed}"] = subset

membership.to_csv(OUT / "split_membership.csv", index=False)
#############################
# Standardize features and target using training nodes only
summary_rows = []
for date in DATES:
    table = pd.read_csv(PREP / f"grid_node_table_{date}.csv", dtype={"grid_id": str}).sort_values("local_node_id").reset_index(drop=True)
    date_dir = OUT / date
    date_dir.mkdir(exist_ok=True)
    table.to_csv(date_dir / "grid_node_table.csv", index=False)
    y_raw = table["lst"].to_numpy(dtype=np.float64)

    for group, feats in FEATURE_GROUPS.items():
        group_dir = date_dir / group
        group_dir.mkdir(exist_ok=True)
        with open(group_dir / "feature_names.json", "w", encoding="utf-8") as f:
            json.dump(feats, f, indent=2)

        X_raw = table[feats].to_numpy(dtype=np.float64)

        for seed in SPLITS:
            masks = masks_by_split[seed]
            train = masks["mask_train"]

            x_mean = X_raw[train].mean(axis=0)
            x_std = X_raw[train].std(axis=0, ddof=0)
            x_std = np.where(x_std < 1e-8, 1.0, x_std)

            y_mean = float(y_raw[train].mean())
            y_std = float(y_raw[train].std(ddof=0))
            if y_std < 1e-8:
                y_std = 1.0

            X = ((X_raw - x_mean) / x_std).astype(np.float32)
            y = ((y_raw - y_mean) / y_std).astype(np.float32)

            np.savez_compressed(
                group_dir / f"split_{seed}.npz",
                X=X, y=y, X_raw=X_raw, y_raw=y_raw,
                x_mean=x_mean, x_std=x_std,
                y_mean=np.asarray(y_mean), y_std=np.asarray(y_std),
                node_ids=common["grid_id"].to_numpy(dtype=str),
                original_node_ids=original_ids,
                feature_names=np.asarray(feats, dtype=str),
                **masks
            )
            #############################################
            summary_rows.append({
                "date": date, "feature_group": group, "split_seed": seed,
                "nodes": n, "features": len(feats),
                "train": int(train.sum()), "val": int(masks["mask_valid"].sum()), "test": int(masks["mask_test"].sum()),
                "train_lst_mean": y_mean, "train_lst_std": y_std,
            })

# Copy summary files and zip backup
for name in [
    "preparation_config.json", "date_quality_summary.csv", "coverage_summary.csv",
    "raster_cleaning_summary.csv", "quality_by_grid_and_date.csv",
    "graph_summary.csv", "common_grid_geometry_wkt.csv",
]:
    if (PREP / name).exists():
        shutil.copy2(PREP / name, OUT / name)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "data_summary.csv", index=False)
shutil.make_archive(str(OUT) + "_backup", "zip", root_dir=OUT.parent, base_dir=OUT.name)
display(summary[summary["feature_group"] == "all"].round(4))

# 8 compass directions neighbor table

In [ ]:
BASE = PROJECT_ROOT / "gnn_outputs"
NODE_FILE = BASE / "grid90_three_dates_preparation_v1" / "common_node_order.csv"
OUT = BASE / "ordered_neighbors_grid90_v1"
OUT.mkdir(parents=True, exist_ok=True)

nodes = pd.read_csv(NODE_FILE, dtype={"grid_id": "string"})
nodes["grid_row"] = nodes["grid_row"].astype(int)
nodes["grid_col"] = nodes["grid_col"].astype(int)

lookup = {(r, c): gid for gid, r, c in zip(nodes["grid_id"], nodes["grid_row"], nodes["grid_col"])}

offsets = {
    "N": (-1, 0), "NE": (-1, 1), "E": (0, 1), "SE": (1, 1),
    "S": (1, 0),  "SW": (1, -1), "W": (0, -1), "NW": (-1, -1),
}

ordered = nodes[["grid_id", "grid_row", "grid_col"]].copy()
positions = list(zip(nodes["grid_row"], nodes["grid_col"]))

for d, (dr, dc) in offsets.items():
    ordered[d] = pd.array([lookup.get((r + dr, c + dc)) for r, c in positions], dtype="string")

ordered["n_neighbors"] = ordered[list(offsets.keys())].notna().sum(axis=1)
ordered.to_csv(OUT / "grid90_ordered_neighbors.csv", index=False, encoding="utf-8-sig")

print("Saved 8-direction neighbors. Total nodes:", len(ordered), "Isolated:", int((ordered["n_neighbors"] == 0).sum()))
display(ordered.head(3))
##################################
def neighbour_indices(table, ids, directions):
    """Convert neighbour IDs to row positions. Missing directions keep index -1."""
    index = pd.Index(ids)
    slots = np.full((len(ids), len(directions)), -1, dtype=np.int64)
    for k, direction in enumerate(directions):
        column = table[direction]
        present = column.notna().to_numpy()
        positions = index.get_indexer(column[present])
        if (positions < 0).any():
            raise ValueError(f"Unknown neighbour ID in direction {direction}.")
        slots[present, k] = positions
    return slots


def ordered_features(center, slots):
    """Stack centre features, eight neighbour blocks and eight presence flags."""
    present = slots >= 0
    blocks = np.full((*slots.shape, center.shape[1]), np.nan, dtype=np.float32)
    blocks[present] = center[slots[present]]
    return np.concatenate([center, blocks.reshape(len(center), -1), present.astype(np.float32)], axis=1)

# 2022 ordered inputs, 80/10/10 splits, and zipping

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

BASE = PROJECT_ROOT / "gnn_outputs"
DATE = "2022-06-17"
GROUP = "public"
FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"]
DIRECTIONS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
SEEDS = [42, 123, 2026, 3407, 777]

OUT_2022 = BASE / "mlp_ordered_grid90_v1" / DATE / GROUP
OUT_2022.mkdir(parents=True, exist_ok=True)

# Build 2022 ordered features
nodes = pd.read_csv(BASE / "grid90_three_dates_preparation_v1" / f"grid_node_table_{DATE}.csv", dtype={"grid_id": "string"})
nodes = nodes.sort_values("local_node_id").reset_index(drop=True)
neighbors = pd.read_csv(BASE / "ordered_neighbors_grid90_v1" / "grid90_ordered_neighbors.csv", dtype="string")
grid_ids = pd.Index(nodes["grid_id"], name="grid_id")
neighbors = neighbors.set_index("grid_id").loc[grid_ids]

X_center = nodes[FEATURES].to_numpy(dtype=np.float32)
y = nodes["lst"].to_numpy(dtype=np.float32)

n = len(nodes)
slots = neighbour_indices(neighbors, grid_ids, DIRECTIONS)
neighbor_mask = slots >= 0
X_ordered = ordered_features(X_center, slots)

ordered_columns = (
    [f"center_{f}" for f in FEATURES]
    + [f"{d}_{f}" for d in DIRECTIONS for f in FEATURES]
    + [f"has_{d}" for d in DIRECTIONS]
)

np.savez_compressed(
    OUT_2022 / "mlp_inputs_raw.npz",
    X_center=X_center, X_ordered=X_ordered, neighbor_mask=neighbor_mask,
    y_lst_C=y, local_node_id=nodes["local_node_id"].to_numpy(dtype=np.int64),
    grid_id=nodes["grid_id"].to_numpy(dtype=str),
)

with open(OUT_2022 / "input_columns.json", "w", encoding="utf-8") as f:
    json.dump({
        "date": DATE, "feature_group": GROUP,
        "center_features": FEATURES, "direction_order": DIRECTIONS,
        "ordered_columns": ordered_columns,
        "target": "LST of the center cell in degrees Celsius",
        "standardized": False,
    }, f, indent=2)
######################################################################
# Create 5 random 80/10/10 splits
SPLIT_OUT = BASE / "mlp_ordered_grid90_v1" / "splits_80_10_10"
SPLIT_OUT.mkdir(parents=True, exist_ok=True)

n_val = round(n * 0.10)
n_test = round(n * 0.10)
n_train = n - n_val - n_test

split_rows = []
for seed in SEEDS:
    order = np.random.default_rng(seed).permutation(n)
    train_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    test_mask = np.zeros(n, dtype=bool)

    train_mask[order[:n_train]] = True
    val_mask[order[n_train:n_train + n_val]] = True
    test_mask[order[n_train + n_val:]] = True
    trainval_mask = train_mask | val_mask

    np.savez_compressed(
        SPLIT_OUT / f"split_{seed}.npz",
        split_seed=seed,
        local_node_id=nodes["local_node_id"].to_numpy(dtype=np.int64),
        grid_id=nodes["grid_id"].to_numpy(dtype=str),
        train_mask=train_mask, val_mask=val_mask, test_mask=test_mask, trainval_mask=trainval_mask,
    )
    split_rows.append({
        "split_seed": seed, "nodes": n,
        "train": int(train_mask.sum()), "val": int(val_mask.sum()),
        "test": int(test_mask.sum()), "final_train": int(trainval_mask.sum()),
    })

split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(SPLIT_OUT / "split_summary.csv", index=False)

# Zip the 2022 package
MLP_BASE = BASE / "mlp_ordered_grid90_v1"
files_to_zip = [
    OUT_2022 / "mlp_inputs_raw.npz",
    OUT_2022 / "input_columns.json",
    SPLIT_OUT / "split_summary.csv",
] + [SPLIT_OUT / f"split_{s}.npz" for s in SEEDS]

zip_path = BASE / "mlp_ordered_grid90_20220617_public_v1.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
    for p in files_to_zip:
        zf.write(p, arcname=p.relative_to(MLP_BASE).as_posix())

print("2022 zip saved to:", zip_path)

# 2023 & 2024 inputs and directional graph

In [ ]:
BASE = PROJECT_ROOT / "gnn_outputs"
TABLE_DIR = BASE / "grid90_three_dates_preparation_v1"
NEIGHBOR_FILE = BASE / "ordered_neighbors_grid90_v1" / "grid90_ordered_neighbors.csv"
MASK_DIR = BASE / "mlp_ordered_grid90_v1" / "splits_80_10_10"
OUT = BASE / "grid90_2023_2024_public_training_v1"

DATES = ["2023-08-15", "2024-08-25"]
FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"]
DIRECTIONS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
SEEDS = [42, 123, 2026, 3407, 777]

# Load node order and 80/10/10 splits
order = pd.read_csv(TABLE_DIR / "common_node_order.csv", dtype={"grid_id": str}).sort_values("local_node_id").reset_index(drop=True)
n = len(order)
ids = order["grid_id"].to_numpy()

masks_by_seed = {}
split_rows = []
for seed in SEEDS:
    z = np.load(MASK_DIR / f"split_{seed}.npz")
    tr, va, te = z["train_mask"], z["val_mask"], z["test_mask"]
    masks_by_seed[seed] = {"train_mask": tr, "val_mask": va, "test_mask": te, "final_train_mask": tr | va}
    split_rows.append({
        "split_seed": seed, "nodes": n,
        "train": int(tr.sum()), "val": int(va.sum()), "test": int(te.sum()),
        "final_train": int((tr | va).sum()),
    })

# Build 8-direction neighbor index matrix and directed edges
nb_table = pd.read_csv(NEIGHBOR_FILE, dtype="string").set_index("grid_id").loc[ids]
nb = neighbour_indices(nb_table, ids, DIRECTIONS)

present = nb >= 0
degree = present.sum(axis=1)
dst, edge_type = np.where(present)
src = nb[dst, edge_type]
edge_index = np.stack([src, dst]).astype(np.int64)
edge_type = edge_type.astype(np.int64)

# Save shared graph and splits
shared = OUT / "shared"
split_dir = shared / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

order.to_csv(shared / "node_order.csv", index=False)
nb_table.reset_index().to_csv(shared / "ordered_neighbors.csv", index=False)
np.save(shared / "neighbor_index.npy", nb)
np.save(shared / "edge_index.npy", edge_index)
np.save(shared / "edge_type.npy", edge_type)

pd.DataFrame({
    "edge_type": range(8),
    "direction": DIRECTIONS,
    "opposite_direction": [DIRECTIONS[(k + 4) % 8] for k in range(8)],
    "valid_neighbors": np.bincount(edge_type, minlength=8),
}).to_csv(shared / "direction_definition.csv", index=False)

for seed, masks in masks_by_seed.items():
    np.savez_compressed(split_dir / f"split_{seed}.npz", **masks)
pd.DataFrame(split_rows).to_csv(shared / "split_summary.csv", index=False)

# Export 2023 and 2024 feature arrays
date_rows = []
for date in DATES:
    t = pd.read_csv(TABLE_DIR / f"grid_node_table_{date}.csv", dtype={"grid_id": str}).set_index("grid_id").loc[ids].reset_index()
    X = t[FEATURES].to_numpy(dtype=np.float32)
    y = t["lst"].to_numpy(dtype=np.float32)
    X_ordered = ordered_features(X, nb)

    date_out = OUT / date
    date_out.mkdir(exist_ok=True)
    np.save(date_out / "X_center_raw.npy", X)
    np.save(date_out / "X_ordered_raw.npy", X_ordered)
    np.save(date_out / "y_lst_C.npy", y)

    tr42 = masks_by_seed[42]["train_mask"]
    date_rows.append({
        "date": date, "nodes": n,
        "center_features": X.shape[1], "ordered_features": X_ordered.shape[1],
        "LST_mean_C": float(y.mean()), "LST_std_C": float(y.std()),
        "LST_min_C": float(y.min()), "LST_max_C": float(y.max()),
        "split42_train_LST_mean_C": float(y[tr42].mean()),
    })

date_summary = pd.DataFrame(date_rows)
date_summary.to_csv(OUT / "date_summary.csv", index=False)

# Save config and zip backup
ordered_columns = (
    [f"center_{f}" for f in FEATURES]
    + [f"{d}_{f}" for d in DIRECTIONS for f in FEATURES]
    + [f"{d}_exists" for d in DIRECTIONS]
)
config = {
    "dates": DATES,
    "spatial_representation": "90 m square grid",
    "feature_group": "public",
    "features": FEATURES,
    "ordered_columns": ordered_columns,
    "directions": DIRECTIONS,
    "nodes": n,
    "undirected_edges": int(edge_index.shape[1] // 2),
    "directed_entries": int(edge_index.shape[1]),
    "isolated_nodes": int((degree == 0).sum()),
    "split_seeds": SEEDS,
    "arrays_are_standardized": False,
}
with open(OUT / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

shutil.make_archive(str(OUT) + "_backup", "zip", root_dir=OUT.parent, base_dir=OUT.name)
display(date_summary.round(4))